# WBC Classification — IMA205 Challenge 2026
### Version finale avec calibration température + poids par classe

## 0 · Configuration

In [1]:
CFG = dict(
    # --- paths ---
    # ⚠️ Vérifier avec : !ls /kaggle/input/
    data_dir = "/kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge",

    backbone     = "convnext_base.fb_in22k_ft_in1k",
    img_size     = 384,

    head_dims    = [768, 384],
    head_dropout = 0.4,

    sampler_alpha = 0.6,
    rare_boost    = {"PLY": 6.0, "PC": 3.0, "PMY": 3.0},

    use_balanced_softmax = True,
    tau = 1.0,

    use_ema    = True,
    ema_decay  = 0.999,

    use_onecycle = True,
    max_lr       = 8e-4,
    pct_start    = 0.1,

    freeze_epochs   = 2,
    backbone_lr_mul = 0.1,

    epochs       = 50,
    batch_size   = 12,
    accum_steps  = 4,
    lr           = 1e-3,
    weight_decay = 1e-4,
    focal_gamma  = 2.0,

    finetune_epochs = 5,
    warmup_epochs   = 3,
    eta_min         = 2e-5,

    hflip_p      = 0.1,
    vflip_p      = 0.1,
    rot_degrees  = 45,
    color_jitter = dict(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.07),
    grayscale_p  = 0.1,
    erasing_p    = 0.1,
    mixup_alpha  = 0.1,
    cutmix_alpha = 0.1,

    tta_n_views = 8,

    seed        = 42,
    num_workers = 2,
    val_frac    = 0.2,

    rare_classes       = {"PLY", "PC", "PMY"},
    rare_tfms_prob     = 0.9,
    ultra_rare_classes = {"PLY", "PC"},
    disable_mix_for_rare = True,
)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print("CFG loaded.")

CFG loaded.


## 1 · Imports & Seeds

In [2]:
import os, random, math, warnings
from datetime import datetime
from collections import defaultdict

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image
from scipy.optimize import minimize_scalar

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import autocast, GradScaler
import torchvision.transforms as T
import torchvision.transforms.functional as TF

import timm
from timm.data import resolve_data_config

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

from tqdm.auto import tqdm
import matplotlib.pyplot as plt

torch.backends.cudnn.benchmark = True

def set_seed(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

set_seed(CFG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {device}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

print(f"timm    : {timm.__version__}")
print(f"Started : {datetime.now():%Y-%m-%d %H:%M:%S}")

Device  : cuda
GPU     : Tesla T4
VRAM    : 15.6 GB
timm    : 1.0.25
Started : 2026-04-03 22:41:03


## 2 · Data Loading

In [3]:
# Vérification du chemin
print("Fichiers disponibles dans /kaggle/input :")
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f.endswith(".csv") or f.endswith(".pth"):
            print(" ", os.path.join(root, f))
    if root.count(os.sep) - "/kaggle/input".count(os.sep) > 2:
        break

Fichiers disponibles dans /kaggle/input :
  /kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/sample_submission.csv
  /kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/train_metadata.csv
  /kaggle/input/competitions/ima205-challenge-2026/IMA205-challenge/test_metadata.csv


In [4]:
TRAIN_DIR = os.path.join(CFG["data_dir"], "train")
TEST_DIR  = os.path.join(CFG["data_dir"], "test")

train_df = pd.read_csv(os.path.join(CFG["data_dir"], "train_metadata.csv"))
test_df  = pd.read_csv(os.path.join(CFG["data_dir"], "test_metadata.csv"))

label_names = sorted(train_df["label"].unique())
label2idx   = {l: i for i, l in enumerate(label_names)}
ULTRA_RARE_IDS = {label2idx[c] for c in CFG["ultra_rare_classes"] if c in label2idx}
idx2label   = {i: l for l, i in label2idx.items()}
NUM_CLASSES = len(label_names)

train_df["y"] = train_df["label"].map(label2idx)
vc = train_df["label"].value_counts()

print(f"Classes : {NUM_CLASSES}")
print(f"Train   : {len(train_df)}  |  Test: {len(test_df)}")
print(f"Imbalance: {vc.max()/vc.min():.0f}x  "
      f"(most={vc.idxmax()}:{vc.max()}  least={vc.idxmin()}:{vc.min()})")
print("\nClass counts:"); print(vc.to_string())

Classes : 13
Train   : 28901  |  Test: 9634
Imbalance: 1183x  (most=SNE:13015  least=PLY:11)

Class counts:
label
SNE    13015
LY      8101
MO      2746
BL      2012
EO       861
MY       441
BA       415
BNE      391
VLY      366
MMY      360
PMY      114
PC        68
PLY       11


In [5]:
tr_df, va_df = train_test_split(
    train_df, test_size=CFG["val_frac"],
    random_state=CFG["seed"], stratify=train_df["label"]
)
print(f"Train: {len(tr_df)}  |  Val: {len(va_df)}")

Train: 23120  |  Val: 5781


## 3 · Transforms

In [6]:
def make_train_tfms(img_size):
    return T.Compose([
        T.Resize((img_size, img_size), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=CFG["hflip_p"]),
        T.RandomVerticalFlip(p=CFG["vflip_p"]),
        T.RandomRotation(CFG["rot_degrees"], interpolation=T.InterpolationMode.BILINEAR),
        T.ColorJitter(**CFG["color_jitter"]),
        T.RandomGrayscale(p=CFG["grayscale_p"]),
        T.RandomAffine(0, translate=(0.08, 0.08), scale=(0.88, 1.12), shear=10,
                       interpolation=T.InterpolationMode.BILINEAR),
        T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.0)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        T.RandomErasing(p=CFG["erasing_p"], scale=(0.02, 0.12)),
    ])

def make_train_tfms_rare(img_size):
    return T.Compose([
        T.Resize((img_size, img_size), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(p=0.5),
        T.RandomVerticalFlip(p=0.5),
        T.RandomRotation(75, interpolation=T.InterpolationMode.BILINEAR),
        T.RandomAffine(0, translate=(0.12, 0.12), scale=(0.80, 1.20), shear=15,
                       interpolation=T.InterpolationMode.BILINEAR),
        T.ColorJitter(brightness=0.45, contrast=0.45, saturation=0.35, hue=0.10),
        T.RandomGrayscale(p=0.15),
        T.GaussianBlur(kernel_size=5, sigma=(0.1, 2.5)),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        T.RandomErasing(p=0.25, scale=(0.02, 0.18)),
    ])

def make_val_tfms(img_size):
    return T.Compose([
        T.Resize((img_size, img_size), interpolation=T.InterpolationMode.BICUBIC),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

def make_tta_tfms(img_size):
    return T.Compose([
        T.Resize((img_size, img_size), interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(0.5),
        T.RandomVerticalFlip(0.5),
        T.RandomRotation(45, interpolation=T.InterpolationMode.BILINEAR),
        T.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        T.ToTensor(),
        T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])

print("Transforms ready.")

Transforms ready.


## 4 · Datasets & DataLoaders

In [7]:
class WBCDataset(Dataset):
    def __init__(self, df, img_dir, tfms, tfms_rare=None, rare_classes=None, rare_prob=1.0):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tfms = tfms
        self.tfms_rare = tfms_rare
        self.rare_classes = set(rare_classes) if rare_classes is not None else set()
        self.rare_prob = rare_prob

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.loc[i]
        name = str(row["ID"])
        if not name.lower().endswith(".png"):
            name += ".png"
        img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        label = row["label"]
        y = int(row["y"])
        if (self.tfms_rare is not None) and (label in self.rare_classes) and (random.random() < self.rare_prob):
            x = self.tfms_rare(img)
        else:
            x = self.tfms(img)
        return x, y


class TTADataset(Dataset):
    """Retourne N vues augmentées par image — utilisé pour val calibration ET test."""
    def __init__(self, df, img_dir, tta_tfms, n_views=8):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.tfms = tta_tfms
        self.n = n_views

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.loc[i]
        name = str(row["ID"])
        if not name.lower().endswith(".png"):
            name += ".png"
        img = Image.open(os.path.join(self.img_dir, name)).convert("RGB")
        views = torch.stack([self.tfms(img) for _ in range(self.n)])
        return views, name   # ← retourne le nom AVEC .png


def make_weighted_sampler(df):
    counts = df["label"].value_counts().to_dict()

    def w(label):
        base = 1.0 / (counts[label] ** CFG["sampler_alpha"])
        if label in CFG["rare_boost"]:
            base *= CFG["rare_boost"][label]
        return base   # ← return bien hors du if

    weights = df["label"].map(w).values
    weights = torch.DoubleTensor(weights)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


kw = dict(num_workers=CFG["num_workers"], pin_memory=True,
          persistent_workers=(CFG["num_workers"] > 0))

train_dl = DataLoader(
    WBCDataset(
        tr_df, TRAIN_DIR,
        tfms=make_train_tfms(CFG["img_size"]),
        tfms_rare=make_train_tfms_rare(CFG["img_size"]),
        rare_classes=CFG["rare_classes"],
        rare_prob=CFG["rare_tfms_prob"],
    ),
    batch_size=CFG["batch_size"],
    sampler=make_weighted_sampler(tr_df),
    **kw
)

val_dl = DataLoader(
    WBCDataset(va_df, TRAIN_DIR, tfms=make_val_tfms(CFG["img_size"])),
    batch_size=CFG["batch_size"] * 2,
    shuffle=False,
    **kw
)

print(f"train_dl: {len(train_dl)} batches  |  val_dl: {len(val_dl)} batches")

train_dl: 1927 batches  |  val_dl: 241 batches


## 5 · Loss

In [8]:
class BalancedSoftmaxLoss(nn.Module):
    """
    Balanced Softmax = softmax(logits + log(prior)).
    Corrige le biais dû à l'imbalance extrême.
    """
    def __init__(self, class_counts):
        super().__init__()
        counts = torch.tensor(class_counts, dtype=torch.float32)
        self.register_buffer("log_prior", torch.log(counts + 1e-12))

    def forward(self, logits, targets):
        log_prior = self.log_prior.to(logits.device)
        adj = logits + log_prior.unsqueeze(0)
        return F.cross_entropy(adj, targets)

counts = tr_df["label"].value_counts()
class_counts = [counts.get(c, 1) for c in label_names]
criterion = BalancedSoftmaxLoss(class_counts).to(device)
print("Using BalancedSoftmaxLoss. Counts[:5]:", class_counts[:5])

Using BalancedSoftmaxLoss. Counts[:5]: [np.int64(332), np.int64(1609), np.int64(313), np.int64(689), np.int64(6480)]


## 6 · Modèle

In [9]:
class WBCModel(nn.Module):
    """
    timm ConvNeXt-Base + custom deep head.
    ConvNeXt utilise LayerNorm (pas BatchNorm) → immunisé au bug BN training-mode.
    """
    def __init__(self, backbone_name, num_classes, head_dims, head_dropout, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained,
            num_classes=0, drop_path_rate=0.2
        )
        in_feats = self.backbone.num_features
        print(f"Backbone: {backbone_name}  |  feature dim: {in_feats}")

        layers = [nn.LayerNorm(in_feats)]
        prev_dim = in_feats
        for hidden in head_dims:
            layers += [nn.Dropout(head_dropout), nn.Linear(prev_dim, hidden), nn.GELU()]
            prev_dim = hidden
        layers += [nn.Dropout(head_dropout / 2), nn.Linear(prev_dim, num_classes)]
        self.head = nn.Sequential(*layers)

        for m in self.head.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)

        n_bb   = sum(p.numel() for p in self.backbone.parameters())
        n_head = sum(p.numel() for p in self.head.parameters())
        print(f"Params  : backbone={n_bb/1e6:.2f}M  head={n_head/1e6:.3f}M  "
              f"total={(n_bb+n_head)/1e6:.2f}M")

    def forward(self, x):
        return self.head(self.backbone(x))

    def freeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(False)
        print("  Backbone FROZEN.")

    def unfreeze_backbone(self):
        for p in self.backbone.parameters(): p.requires_grad_(True)
        print("  Backbone UNFROZEN.")

    def set_train_mode(self):
        """Head en train, backbone en eval — évite la corruption des stats BN."""
        self.train()
        self.backbone.eval()


model = WBCModel(CFG["backbone"], NUM_CLASSES, CFG["head_dims"], CFG["head_dropout"])
model = model.to(device)

with torch.no_grad():
    _x   = torch.randn(2, 3, CFG["img_size"], CFG["img_size"]).to(device)
    _out = model(_x)
print(f"Output shape: {_out.shape}  ✓")

model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]

Backbone: convnext_base.fb_in22k_ft_in1k  |  feature dim: 1024
Params  : backbone=87.57M  head=1.090M  total=88.66M
Output shape: torch.Size([2, 13])  ✓


## 7 · MixUp / CutMix

In [10]:
def mixup(x, y, alpha):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam*x + (1-lam)*x[idx], y, y[idx], lam

def cutmix(x, y, alpha):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape
    rh, rw  = int(H*math.sqrt(1-lam)), int(W*math.sqrt(1-lam))
    cy, cx  = random.randint(0, H), random.randint(0, W)
    y1, y2  = max(0, cy-rh//2), min(H, cy+rh//2)
    x1, x2  = max(0, cx-rw//2), min(W, cx+rw//2)
    mixed   = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (y2-y1)*(x2-x1)/(H*W)
    return mixed, y, y[idx], lam

def mix_loss(criterion, logits, ya, yb, lam):
    return lam*criterion(logits, ya) + (1-lam)*criterion(logits, yb)

print("MixUp/CutMix ready.")

MixUp/CutMix ready.


## 8 · Optimizer, Scheduler, EMA

In [11]:
def make_optimizer(model):
    return optim.AdamW(
        [{"params": model.head.parameters(),     "lr": CFG["max_lr"],                         "name": "head"},
         {"params": model.backbone.parameters(), "lr": CFG["max_lr"]*CFG["backbone_lr_mul"],  "name": "backbone"}],
        weight_decay=CFG["weight_decay"], betas=(0.9, 0.999)
    )

def make_scheduler_onecycle(optimizer, steps_per_epoch, epochs):
    return optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=[CFG["max_lr"], CFG["max_lr"]*CFG["backbone_lr_mul"]],
        steps_per_epoch=steps_per_epoch,
        epochs=epochs,
        pct_start=CFG["pct_start"],
        div_factor=10.0,
        final_div_factor=100.0,
        anneal_strategy="cos"
    )


class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {k: v.detach().clone() for k, v in model.state_dict().items()}

    @torch.no_grad()
    def update(self, model):
        msd = model.state_dict()
        for k in self.shadow:
            self.shadow[k].mul_(self.decay).add_(msd[k].detach(), alpha=1 - self.decay)

    def copy_to(self, model):
        model.load_state_dict(self.shadow, strict=True)


ema = EMA(model, decay=CFG["ema_decay"]) if CFG["use_ema"] else None
print("Optimizer / Scheduler / EMA ready.")

Optimizer / Scheduler / EMA ready.


## 9 · Train & Validate

In [12]:
scaler = GradScaler()

def train_epoch(model, loader, optimizer, criterion, scheduler, epoch, phase_full=False):
    if phase_full:
        model.train()
    else:
        model.set_train_mode()

    optimizer.zero_grad(set_to_none=True)
    losses = []
    bar = tqdm(enumerate(loader), total=len(loader), desc=f"[Train] E{epoch}", leave=True)

    for step, (x, y) in bar:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        has_ultra_rare = False
        if CFG.get("disable_mix_for_rare", True) and len(ULTRA_RARE_IDS) > 0:
            has_ultra_rare = any((y == rid).any().item() for rid in ULTRA_RARE_IDS)

        if has_ultra_rare:
            ya, yb, lam = y, y, 1.0
        else:
            r = random.random()
            if   r < 0.33 and CFG["mixup_alpha"]  > 0: x, ya, yb, lam = mixup(x, y, CFG["mixup_alpha"])
            elif r < 0.66 and CFG["cutmix_alpha"] > 0: x, ya, yb, lam = cutmix(x, y, CFG["cutmix_alpha"])
            else: ya, yb, lam = y, y, 1.0

        with autocast():
            logits = model(x)
            loss   = mix_loss(criterion, logits, ya, yb, lam) / CFG["accum_steps"]

        scaler.scale(loss).backward()

        if (step+1) % CFG["accum_steps"] == 0 or (step+1) == len(loader):
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            if scheduler is not None: scheduler.step()
            if ema is not None: ema.update(model)

        losses.append(loss.detach().item() * CFG["accum_steps"])
        bar.set_postfix(avg_loss=f"{np.mean(losses):.4f}")

    return float(np.mean(losses))


@torch.no_grad()
def val_epoch(model, loader, criterion, epoch):
    model.eval()

    if ema is not None:
        backup = {k: v.detach().clone() for k, v in model.state_dict().items()}
        ema.copy_to(model)

    losses, all_y, all_p = [], [], []

    for x, y in tqdm(loader, desc=f"[Val ] E{epoch}", leave=True):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        with autocast():
            logits = model(x)
            loss   = criterion(logits, y)
        preds = logits.argmax(1)
        losses.append(loss.item())
        all_y.extend(y.detach().cpu().numpy().tolist())
        all_p.extend(preds.detach().cpu().numpy().tolist())

    avg_loss = float(np.mean(losses))
    macro_f1 = float(f1_score(all_y, all_p, average="macro", zero_division=0))
    avg_acc  = float((np.array(all_y) == np.array(all_p)).mean())
    pc_f1 = f1_score(all_y, all_p, labels=list(range(NUM_CLASSES)),
                     average=None, zero_division=0).astype(np.float32)

    if ema is not None:
        model.load_state_dict(backup, strict=True)

    return avg_loss, macro_f1, avg_acc, pc_f1

print("train_epoch / val_epoch ready.")

train_epoch / val_epoch ready.


## 10 · Boucle d'entraînement principale

In [13]:
hist = defaultdict(list)
best_f1, best_epoch, best_state = -1.0, 1, None

if CFG["freeze_epochs"] > 0:
    model.freeze_backbone()

optimizer = make_optimizer(model)
steps_per_epoch = math.ceil(len(train_dl) / CFG["accum_steps"])
scheduler = make_scheduler_onecycle(optimizer, steps_per_epoch, epochs=CFG["epochs"])

for epoch in range(1, CFG["epochs"] + 1):

    if epoch == CFG["freeze_epochs"] + 1 and CFG["freeze_epochs"] > 0:
        model.unfreeze_backbone()
        optimizer = make_optimizer(model)
        remaining_epochs = CFG["epochs"] - (epoch - 1)
        scheduler = make_scheduler_onecycle(optimizer, steps_per_epoch, remaining_epochs)

    phase_full = (epoch > CFG["freeze_epochs"])
    tr_loss = train_epoch(model, train_dl, optimizer, criterion, scheduler, epoch, phase_full=phase_full)
    va_loss, va_f1, va_acc, pc_f1 = val_epoch(model, val_dl, criterion, epoch)

    hist["tr_loss"].append(tr_loss)
    hist["va_loss"].append(va_loss)
    hist["va_f1"].append(va_f1)
    hist["va_acc"].append(va_acc)
    hist["pc_f1"].append(pc_f1.tolist())

    pc_line = "  ".join([f"{label_names[i]}={pc_f1[i]:.3f}" for i in range(NUM_CLASSES)])
    print(f"[Epoch {epoch:02d}] val_loss={va_loss:.4f}  acc={va_acc:.4f}  macroF1={va_f1:.4f}")
    print(f"           per-class F1: {pc_line}")

    if va_f1 > best_f1:
        best_f1, best_epoch = va_f1, epoch
        if ema is not None:
            best_state = {k: v.detach().cpu().clone() for k, v in ema.shadow.items()}
        else:
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"  ** NEW BEST  macro-F1={best_f1:.4f}  @ epoch {best_epoch}")

print(f"\n{'='*65}")
print(f"  Done.  Best val macro-F1 = {best_f1:.4f}  @ epoch {best_epoch}")
print(f"{'='*65}")

model.load_state_dict(best_state)
torch.save(best_state, "wbc_best.pth")
print("Saved -> wbc_best.pth")

  Backbone FROZEN.


[Train] E1:   0%|          | 0/1927 [00:00<?, ?it/s]

[Val ] E1:   0%|          | 0/241 [00:00<?, ?it/s]

[Epoch 01] val_loss=1.3719  acc=0.0031  macroF1=0.0204
           per-class F1: BA=0.000  BL=0.000  BNE=0.007  EO=0.000  LY=0.000  MMY=0.000  MO=0.000  MY=0.000  PC=0.019  PLY=0.001  PMY=0.239  SNE=0.000  VLY=0.000
  ** NEW BEST  macro-F1=0.0204  @ epoch 1


[Train] E2:   0%|          | 0/1927 [00:00<?, ?it/s]

[Val ] E2:   0%|          | 0/241 [00:00<?, ?it/s]

... (141 outputs left out)

[Epoch 49] val_loss=0.5131  acc=0.9353  macroF1=0.6847
           per-class F1: BA=0.957  BL=0.908  BNE=0.395  EO=0.965  LY=0.959  MMY=0.586  MO=0.912  MY=0.693  PC=0.571  PLY=0.000  PMY=0.400  SNE=0.977  VLY=0.578


[Train] E50:   0%|          | 0/1927 [00:00<?, ?it/s]

[Val ] E50:   0%|          | 0/241 [00:00<?, ?it/s]

[Epoch 50] val_loss=0.5155  acc=0.9353  macroF1=0.6847
           per-class F1: BA=0.957  BL=0.907  BNE=0.395  EO=0.965  LY=0.959  MMY=0.586  MO=0.913  MY=0.693  PC=0.571  PLY=0.000  PMY=0.400  SNE=0.977  VLY=0.578

  Done.  Best val macro-F1 = 0.7419  @ epoch 37
Saved -> wbc_best.pth


## 11 · Post-training : Calibration Température + Poids par classe

Cette étape **ne nécessite pas de ré-entraînement**.
Elle optimise les prédictions sur le val set pour maximiser le macro-F1.

In [14]:
# Récupérer les logits sur le val set avec le meilleur modèle (EMA)
model.load_state_dict(best_state)
model.eval()

@torch.inference_mode()
def get_logits_labels(mdl, loader):
    mdl.eval()
    all_logits, all_y = [], []
    for x, y in tqdm(loader, desc="[Val logits]"):
        x = x.to(device)
        with autocast():
            logits = mdl(x)
        all_logits.append(logits.cpu())
        all_y.extend(y.tolist())
    return torch.cat(all_logits), np.array(all_y)

logits_val, labels_val = get_logits_labels(model, val_dl)

# F1 de base (sans calibration)
base_preds = logits_val.softmax(-1).argmax(-1).numpy()
base_f1 = f1_score(labels_val, base_preds, average="macro", zero_division=0)
print(f"Val F1 de base (sans calibration) : {base_f1:.4f}")

[Val logits]:   0%|          | 0/241 [00:00<?, ?it/s]

Val F1 de base (sans calibration) : 0.7419


In [15]:
# --- Temperature Scaling ---
def neg_f1_T(T):
    preds = (logits_val / T).softmax(-1).argmax(-1).numpy()
    return -f1_score(labels_val, preds, average="macro", zero_division=0)

res = minimize_scalar(neg_f1_T, bounds=(0.3, 3.0), method="bounded")
best_T = res.x
print(f"Best Temperature : {best_T:.4f}  →  Val F1 = {-res.fun:.4f}")

Best Temperature : 3.0000  →  Val F1 = 0.7419


In [16]:
# --- Per-class weight optimization ---
base_probs = (logits_val / best_T).softmax(-1).numpy()
best_weights = np.ones(NUM_CLASSES)

for c in range(NUM_CLASSES):
    best_f1_c, best_w = -1, 1.0
    for w in np.linspace(0.3, 8.0, 80):
        adj = base_probs.copy()
        adj[:, c] *= w
        preds = adj.argmax(axis=1)
        f = f1_score(labels_val, preds, average="macro", zero_division=0)
        if f > best_f1_c:
            best_f1_c, best_w = f, w
    best_weights[c] = best_w
    print(f"  {label_names[c]:4s}: weight={best_w:.2f}  →  val_F1={best_f1_c:.4f}")

# Vérification globale
adj_probs = base_probs * best_weights[np.newaxis, :]
final_f1 = f1_score(labels_val, adj_probs.argmax(axis=1), average="macro", zero_division=0)
print(f"\n✅ Val macro-F1 après calibration complète : {final_f1:.4f}")
print(f"   Gain vs base : +{final_f1 - base_f1:.4f}")

  BA  : weight=1.37  →  val_F1=0.7420
  BL  : weight=1.08  →  val_F1=0.7423
  BNE : weight=1.37  →  val_F1=0.7465
  EO  : weight=0.49  →  val_F1=0.7426
  LY  : weight=1.18  →  val_F1=0.7429
  MMY : weight=1.08  →  val_F1=0.7429
  MO  : weight=0.69  →  val_F1=0.7424
  MY  : weight=0.40  →  val_F1=0.7423
  PC  : weight=2.83  →  val_F1=0.7428
  PLY : weight=0.30  →  val_F1=0.7419
  PMY : weight=0.88  →  val_F1=0.7419
  SNE : weight=2.15  →  val_F1=0.7446
  VLY : weight=0.69  →  val_F1=0.7435

✅ Val macro-F1 après calibration complète : 0.7399
   Gain vs base : +-0.0019


## 12 · TTA Inference sur le test set

In [17]:
test_ds = TTADataset(
    test_df, TEST_DIR,
    make_tta_tfms(CFG["img_size"]),
    n_views=CFG["tta_n_views"]
)
test_dl_inf = DataLoader(
    test_ds,
    batch_size=max(1, CFG["batch_size"] // 4),
    shuffle=False,
    num_workers=CFG["num_workers"],
    pin_memory=True,
    persistent_workers=(CFG["num_workers"] > 0)
)

@torch.inference_mode()
def predict_tta(mdl, loader, temperature, weights):
    mdl.eval()
    all_ids, all_probs = [], []

    for views, ids in tqdm(loader, desc="[TTA Inference]"):
        B, N, C, H, W = views.shape
        x = views.view(B*N, C, H, W).to(device)
        with autocast():
            logits = mdl(x)
            probs  = F.softmax(logits / temperature, dim=1)
        probs = probs.view(B, N, -1).mean(1)
        all_probs.append(probs.cpu())
        all_ids.extend(list(ids))

    all_probs = torch.cat(all_probs).numpy()
    # Appliquer les poids par classe
    all_probs = all_probs * weights[np.newaxis, :]
    return all_ids, all_probs


pred_ids, pred_probs = predict_tta(model, test_dl_inf, best_T, best_weights)
print(f"Predictions done : {len(pred_ids)} images")

[TTA Inference]:   0%|          | 0/3212 [00:00<?, ?it/s]

Predictions done : 9634 images


## 13 · Création du fichier submission

In [18]:
pred_labels = [idx2label[int(i)] for i in pred_probs.argmax(axis=1)]

# ✅ FORMAT CORRECT :
#   - ID avec .png (déjà inclus par TTADataset)
#   - Pas de colonne "Unnamed: 0"
#   - index=False dans to_csv
sub = pd.DataFrame({
    "ID":    pred_ids,     # ex: "00001.png"
    "label": pred_labels
})

sub.to_csv("submission.csv", index=False)

print(f"✅ Saved -> submission.csv  shape={sub.shape}")
print(sub.head(10))
print("\nDistribution des prédictions:")
print(sub["label"].value_counts())

✅ Saved -> submission.csv  shape=(9634, 2)
               ID label
0  test_00000.png    LY
1  test_00001.png    BL
2  test_00002.png   SNE
3  test_00003.png   SNE
4  test_00004.png    LY
5  test_00005.png    MO
6  test_00006.png   SNE
7  test_00007.png    LY
8  test_00008.png    EO
9  test_00009.png    BA

Distribution des prédictions:
label
SNE    4310
LY     2717
MO      889
BL      682
EO      280
MMY     163
BNE     145
BA      143
MY      141
VLY     103
PMY      40
PC       19
PLY       2
Name: count, dtype: int64


## 14 · (Optionnel) Relancer uniquement l'inférence depuis wbc_best.pth

Si tu as déjà un `wbc_best.pth` sauvegardé dans un run précédent,
tu peux sauter toute la partie entraînement et aller directement ici.

In [19]:
# ============================================================
# DÉCOMMENTER CE BLOC pour repartir d'un checkpoint existant
# sans réentraîner (après avoir ajouté wbc_best.pth en Input)
# ============================================================

# # Trouver le chemin du modèle
# for root, dirs, files in os.walk("/kaggle/input"):
#     for f in files:
#         if f.endswith(".pth"):
#             print(os.path.join(root, f))

# MODEL_PATH = "/kaggle/input/TON-DATASET/wbc_best.pth"  # ← adapter

# model_inf = WBCModel(
#     CFG["backbone"], NUM_CLASSES, CFG["head_dims"], CFG["head_dropout"],
#     pretrained=False  # pas besoin de télécharger ImageNet
# ).to(device)
# model_inf.load_state_dict(torch.load(MODEL_PATH, map_location=device))
# model_inf.eval()
# print("Model loaded. ✓")

# # Calibration sur val set
# logits_val, labels_val = get_logits_labels(model_inf, val_dl)
# res = minimize_scalar(lambda T: -f1_score(
#     labels_val,
#     (logits_val/T).softmax(-1).argmax(-1).numpy(),
#     average="macro", zero_division=0
# ), bounds=(0.3, 3.0), method="bounded")
# best_T = res.x
# base_probs = (logits_val / best_T).softmax(-1).numpy()
# best_weights = np.ones(NUM_CLASSES)
# for c in range(NUM_CLASSES):
#     best_f1_c, best_w = -1, 1.0
#     for w in np.linspace(0.3, 8.0, 80):
#         adj = base_probs.copy(); adj[:, c] *= w
#         f = f1_score(labels_val, adj.argmax(1), average="macro", zero_division=0)
#         if f > best_f1_c: best_f1_c, best_w = f, w
#     best_weights[c] = best_w
# print(f"T={best_T:.3f}  weights={best_weights}")

# # Inférence + submission
# pred_ids, pred_probs = predict_tta(model_inf, test_dl_inf, best_T, best_weights)
# pred_labels = [idx2label[int(i)] for i in pred_probs.argmax(axis=1)]
# sub = pd.DataFrame({"ID": pred_ids, "label": pred_labels})
# sub.to_csv("submission.csv", index=False)
# print(f"✅ submission.csv  shape={sub.shape}")
# print(sub.head())

print("Cellule optionnelle — décommenter si besoin.")

Cellule optionnelle — décommenter si besoin.
